In [15]:
import numpy as np
import torch
import random
import os
import cv2
import copy
import logging
import math
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
import time
import warnings
warnings.filterwarnings("ignore")
import ml_collections
import math
import weakref
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
import torch.nn.functional as F
from torchvision.transforms import v2
from torchvision import tv_tensors
import pandas as pd


In [2]:
from scipy.ndimage import zoom
from torch.utils.data import Dataset
from torchvision import transforms as T
from torchvision.transforms import functional as F
from typing import Callable
from PIL import Image
from scipy import ndimage

from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

from torch.nn import Dropout, Softmax, Conv2d, LayerNorm
from torch.nn.modules.utils import _pair

from utils import *

from sklearn.metrics import roc_auc_score,jaccard_score
from torch import nn
from functools import wraps
from torch.optim.optimizer import Optimizer
from sklearn.metrics import roc_auc_score,jaccard_score
from torch import nn
from functools import wraps
from torch.optim.optimizer import Optimizer
from torch.utils.tensorboard import SummaryWriter
from torch.backends import cudnn
from torch.utils.data import DataLoader
from torch.utils.data import DataLoader
from tqdm import tqdm
from utils import *
from torch.backends import cudnn
from torchvision import transforms

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cuda


In [4]:
class SurfaceDefectDatasetV2(Dataset):
    """
    Tối ưu hóa Data Pipeline với torchvision.transforms.v2.
    Hỗ trợ linh hoạt cả ảnh Xám (Grayscale) và ảnh Màu (RGB) với CLAHE.
    """
    def __init__(self, data_path, type='train', transform=False, use_grayscale=True, apply_clahe=True):
        super().__init__()
        
        # 1. Tải dữ liệu từ file .npz
        data_np = np.load(data_path)
        self.images = data_np[f"x_{type}"]
        self.masks  = data_np[f"y_{type}"]
        
        self.transform = transform
        self.use_grayscale = use_grayscale
        self.apply_clahe = apply_clahe
        
        # Khởi tạo CLAHE nếu được yêu cầu
        if self.apply_clahe:
            self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            
        # 2. Cấu hình Normalize động dựa trên số lượng kênh màu
        # Nếu là ảnh xám (1 kênh) -> mean/std có 1 giá trị
        # Nếu là ảnh màu (3 kênh) -> mean/std có 3 giá trị
        mean = [0.5] if self.use_grayscale else [0.5, 0.5, 0.5]
        std  = [0.5] if self.use_grayscale else [0.5, 0.5, 0.5]

        # 3. Xây dựng luồng Augmentation V2
        if self.transform:
            self.aug_pipeline = v2.Compose([
                v2.RandomHorizontalFlip(p=0.5),
                v2.RandomVerticalFlip(p=0.5),
                # Xoay 90, 180, 270 độ (không mất mát dữ liệu)
                v2.RandomChoice([
                    v2.RandomRotation(degrees=[90, 90]),
                    v2.RandomRotation(degrees=[180, 180]),
                    v2.RandomRotation(degrees=[270, 270]),
                ], p=[0.25, 0.25, 0.25]), 
                # Xoay góc nhỏ (-15 đến 15 độ), V2 sẽ tự động dùng BILINEAR cho ảnh, NEAREST cho mask
                v2.RandomRotation(degrees=[-15, 15], interpolation=v2.InterpolationMode.BILINEAR),
                v2.Normalize(mean=mean, std=std)
            ])
        else:
            self.aug_pipeline = v2.Compose([
                v2.Normalize(mean=mean, std=std)
            ])

    def preprocess_image(self, img_array):
        """Tiền xử lý thông minh hỗ trợ đa dạng dữ liệu"""
        # Trường hợp 1: Chuyển về ảnh xám (Ví dụ: Tập NEU-Seg)
        if self.use_grayscale:
            if len(img_array.shape) == 3 and img_array.shape[2] == 3:
                img_array = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
            if self.apply_clahe:
                img_array = self.clahe.apply(img_array)
                
        # Trường hợp 2: Giữ nguyên ảnh màu (Ví dụ: Bề mặt trái cây, vải vóc, gỉ sét)
        else:
            if self.apply_clahe and len(img_array.shape) == 3:
                # Chuyển sang không gian LAB để áp dụng CLAHE lên kênh độ sáng (L)
                lab = cv2.cvtColor(img_array, cv2.COLOR_RGB2LAB)
                l, a, b = cv2.split(lab)
                l_clahe = self.clahe.apply(l)
                lab_clahe = cv2.merge((l_clahe, a, b))
                img_array = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2RGB)
                
        return img_array

    def __getitem__(self, idx):
        # 1. Tiền xử lý bằng OpenCV/Numpy
        img_np = self.preprocess_image(self.images[idx])
        msk_np = self.masks[idx]

        # 2. Định hình lại ma trận (Reshape) cho chuẩn PyTorch (Channels, Height, Width)
        if img_np.ndim == 2:
            img_np = np.expand_dims(img_np, axis=0)  # (1, H, W)
        elif img_np.ndim == 3:
            img_np = img_np.transpose((2, 0, 1))     # (C, H, W)

        if msk_np.ndim == 2:
            msk_np = np.expand_dims(msk_np, axis=0)  # (1, H, W)

        # 3. Chuyển sang Tensor và đưa giá trị về dải [0.0, 1.0]
        img_tensor = torch.from_numpy(img_np).float() / 255.0
        msk_tensor = torch.from_numpy(msk_np).float() / 255.0

        # 4. Đóng gói bằng tv_tensors để kích hoạt sức mạnh đồng bộ của V2
        img = tv_tensors.Image(img_tensor)
        msk = tv_tensors.Mask(msk_tensor)

        # 5. Chạy qua Pipeline Augmentation
        img, msk = self.aug_pipeline(img, msk)

        return img, msk

    def __len__(self):
        return len(self.images)

In [5]:
# 1. CÁC KHỐI TÍCH CHẬP LÕI (CORE BLOCKS)
# ==========================================
class XConv(nn.Module):
    """Khối tích chập hình chữ X 7x7 (Đã tối ưu bằng Mask)"""
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(
            channels, channels, 
            kernel_size=7, 
            padding=3, 
            groups=channels, 
            bias=False
        )
        
        # Tạo mặt nạ (mask) hình chữ X
        mask = torch.zeros(7, 7)
        for i in range(7):
            mask[i, i] = 1         # Đường chéo chính
            mask[i, 6 - i] = 1     # Đường chéo phụ
            
        mask = mask.view(1, 1, 7, 7).expand(channels, 1, 7, 7)
        self.register_buffer('mask', mask)

    def forward(self, x):
        # Chỉ giữ lại trọng số trên hình chữ X
        masked_weight = self.conv.weight * self.mask
        out = torch.nn.functional.conv2d(
            x, 
            weight=masked_weight, 
            bias=self.conv.bias, 
            stride=self.conv.stride, 
            padding=self.conv.padding, 
            groups=self.conv.groups
        )
        return out

class AxialDW(nn.Module):
    """Khối Axial Depthwise: Cộng song song 2 nhánh 1x7 và 7x1"""
    def __init__(self, channels):
        super().__init__()
        self.dw_1x7 = nn.Conv2d(channels, channels, kernel_size=(1, 7), padding=(0, 3), groups=channels, bias=False)
        self.dw_7x1 = nn.Conv2d(channels, channels, kernel_size=(7, 1), padding=(3, 0), groups=channels, bias=False)

    def forward(self, x):
        return self.dw_1x7(x) + self.dw_7x1(x)

In [6]:
# 2. CÁC KHỐI KIẾN TRÚC (ARCHITECTURE BLOCKS)
# ==========================================
class EncoderBlock(nn.Module):
    """Khối Mã hóa (Encoding) và Thu nhỏ ảnh (Downsampling)"""
    def __init__(self, in_c, out_c):
        super().__init__()
        self.xconv = XConv(in_c)
        self.bn = nn.BatchNorm2d(in_c)
        self.pw = nn.Conv2d(in_c, out_c, kernel_size=1, bias=False)
        self.down = nn.MaxPool2d(kernel_size=2, stride=2)
        self.act = nn.GELU()

    def forward(self, x):
        # Đường vòng Residual cộng ngay trước lớp BN
        skip = self.bn(self.xconv(x) + x)
        out = self.act(self.down(self.pw(skip)))
        return out, skip

class DecoderBlock(nn.Module):
    """Khối Giải mã (Decoding), Phóng to (Upsampling) và Ghép nối"""
    def __init__(self, in_c, skip_c, out_c):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.bn = nn.BatchNorm2d(in_c)
        self.pw1 = nn.Conv2d(in_c + skip_c, out_c, kernel_size=1, bias=False)
        self.xconv = XConv(out_c)
        self.pw2 = nn.Conv2d(out_c, out_c, kernel_size=1, bias=False)
        self.act = nn.GELU()

    def forward(self, x, skip):
        x = self.bn(self.up(x))
        x = torch.cat([x, skip], dim=1)  # Ghép nối (Concat) với nhánh skip
        x = self.pw1(x)
        x = self.xconv(x) + x            # Đường vòng Residual bao lấy khối XConv
        out = self.act(self.pw2(x))
        return out

class MultiScaleFusionBlock(nn.Module):
    """Khối Đáy (Bottleneck) - Trích xuất đặc trưng đa tỷ lệ"""
    def __init__(self, in_c, out_c):
        super().__init__()
        mid_c = out_c 
        self.pw1 = nn.Conv2d(in_c, mid_c, kernel_size=1, bias=False)
        
        self.branch1 = XConv(mid_c)
        self.branch2 = AxialDW(mid_c)
        self.branch3 = nn.Conv2d(mid_c, mid_c, kernel_size=3, padding=1, groups=mid_c, bias=False)
        
        self.bn = nn.BatchNorm2d(mid_c * 4)
        self.pw2 = nn.Conv2d(mid_c * 4, out_c, kernel_size=1, bias=False)
        self.act = nn.GELU()

    def forward(self, x):
        x = self.pw1(x)
        
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = x  # Nhánh Identity (đường đi thẳng)
        
        out = torch.cat([b1, b2, b3, b4], dim=1)
        
        out = self.bn(out)
        out = self.pw2(out)
        out = self.act(out)
        return out

In [7]:
# 3. MÔ HÌNH TỔNG THỂ (X-Enhanced ULite)
# ==========================================
class XULite(nn.Module):
    def __init__(self, img_channels=1, num_classes=1, base_c=32):
        """
        img_channels: 1 nếu dùng ảnh Xám, 3 nếu dùng ảnh Màu (RGB)
        num_classes: 1 cho bài toán phân vùng nhị phân (lỗi và nền)
        base_c: Số kênh bắt đầu (quyết định độ lớn của toàn bộ mạng)
        """
        super().__init__()
        
        # Stem: Khối mở đầu để tiếp nhận ảnh và tăng số chiều
        self.stem = nn.Sequential(
            nn.Conv2d(img_channels, base_c, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_c),
            nn.GELU()
        )

        # Nhánh Mã hóa (Encoder)
        self.enc1 = EncoderBlock(in_c=base_c, out_c=base_c * 2)      # 32 -> 64
        self.enc2 = EncoderBlock(in_c=base_c * 2, out_c=base_c * 4)  # 64 -> 128
        self.enc3 = EncoderBlock(in_c=base_c * 4, out_c=base_c * 8)  # 128 -> 256

        # Khối Đáy (Bottleneck)
        self.bottleneck = MultiScaleFusionBlock(in_c=base_c * 8, out_c=base_c * 8) # 256 -> 256

        # Nhánh Giải mã (Decoder)
        self.dec3 = DecoderBlock(in_c=base_c * 8, skip_c=base_c * 4, out_c=base_c * 4) # 256, 128 -> 128
        self.dec2 = DecoderBlock(in_c=base_c * 4, skip_c=base_c * 2, out_c=base_c * 2) # 128, 64  -> 64
        self.dec1 = DecoderBlock(in_c=base_c * 2, skip_c=base_c, out_c=base_c)         # 64, 32   -> 32

        # Lớp xuất kết quả (Classifier)
        self.out_conv = nn.Conv2d(base_c, num_classes, kernel_size=1)

    def forward(self, x):
        # 1. Qua khối mở đầu
        x = self.stem(x)

        # 2. Đi xuống (Thu nhỏ)
        x, skip1 = self.enc1(x)
        x, skip2 = self.enc2(x)
        x, skip3 = self.enc3(x)

        # 3. Qua đáy
        x = self.bottleneck(x)

        # 4. Đi lên (Phóng to)
        x = self.dec3(x, skip3)
        x = self.dec2(x, skip2)
        x = self.dec1(x, skip1)

        # 5. Xuất ảnh dự đoán
        logits = self.out_conv(x)
        return logits

In [20]:
# Giả lập một batch gồm 2 ảnh xám kích thước 256x256
dummy_input = torch.randn(2, 1, 256, 256)
model = XULite(img_channels=1, num_classes=1, base_c=32)

output = model(dummy_input)
print(f"Kích thước ảnh đầu vào: {dummy_input.shape}")
print(f"Kích thước Mask dự đoán:{output.shape}")

Kích thước ảnh đầu vào: torch.Size([2, 1, 256, 256])
Kích thước Mask dự đoán:torch.Size([2, 1, 256, 256])


In [ ]:
# import pandas as pd

# # Sửa lại đường dẫn này tới đúng thư mục chứa file NEU_test_files.csv của bạn
# test_csv_path = r"C:\Users\thuth\Downloads\dseg_models\data\NEU_trainval_test.csv"

# df_test = pd.read_csv(test_csv_path)

# # Đếm luôn số lượng tên ảnh không trùng lặp trong cột ImageId
# total_test_images = df_test['ImageId'].nunique()

# print(f"Tổng số ảnh trong tập Test là: {total_test_images} ảnh.")

Tổng số ảnh trong tập Test là: 3630 ảnh.


In [17]:
# Global Configurations
seed = 666
cosineLR = False
n_channels = 1   # Grayscale input for XULite
n_labels = 1     # Binary mask prediction
epochs = 2000
img_size = 200   
weight_decay = 1e-4
print_frequency = 20 # Tần suất in thông tin huấn luyện (mỗi 20 batch) (3630/32)
save_frequency = 500 # Tần suất lưu mô hình (mỗi 500 batch)
vis_frequency = 10
early_stopping_patience = 50
learning_rate = 1e-3

task_name = 'NEU-seg' 

if task_name in ['NEU-seg']:
    batch_size = 32
else :
    batch_size = 4


base = r"C:\Users\thuth\Downloads\XUlite\datasets"
train_csv_path = os.path.join(base, task_name, "TrainingData", "NEU_train.csv")
val_csv_path = os.path.join(base, task_name, "ValData", "NEU_val.csv")
test_csv_path = os.path.join(base, task_name, "TestData", "NEU_test.csv")

model_name = 'XULite' 
session_name       = 'Test_session' + '_' + time.strftime('%m.%d_%Hh%M')
save_path          = task_name +'/'+ model_name +'/' + session_name + '/'
model_path         = save_path + 'models/'
tensorboard_folder = save_path + 'tensorboard_logs/'
logger_path        = save_path + session_name + ".log"
visualize_path     = save_path + 'visualize_val/'

In [9]:
# 2. DATASETS (HỖ TRỢ ĐA ĐỊNH DẠNG & CLAHE)
# ==========================================
class BaseDefectDataset(Dataset):
    """Class cơ sở chứa các hàm tiền xử lý dùng chung (CLAHE, đổi hệ màu)"""
    def __init__(self, apply_clahe, use_grayscale):
        self.apply_clahe = apply_clahe
        self.use_grayscale = use_grayscale
        if self.apply_clahe:
            self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    def apply_preprocessing(self, img_path):
        img_cv = cv2.imread(img_path)
        if img_cv is None:
            raise FileNotFoundError(f"Không thể đọc ảnh: {img_path}")

        if self.use_grayscale:
            if len(img_cv.shape) == 3:
                img_cv = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
            if self.apply_clahe:
                img_cv = self.clahe.apply(img_cv)
        else:
            if self.apply_clahe and len(img_cv.shape) == 3:
                lab = cv2.cvtColor(img_cv, cv2.COLOR_BGR2LAB)
                l, a, b = cv2.split(lab)
                l_clahe = self.clahe.apply(l)
                lab_clahe = cv2.merge((l_clahe, a, b))
                img_cv = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2RGB)
            else:
                img_cv = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
                
        # Trả về PIL Image để Torchvision xử lý tiếp
        return Image.fromarray(img_cv)

class CSVDefectDataset(BaseDefectDataset):
    """Class xử lý dữ liệu Mask dạng chuỗi RLE trong file CSV"""
    def __init__(self, data_dir, csv_path, original_shape, transforms=None, **kwargs):
        super().__init__(**kwargs)
        self.img_dir = os.path.join(data_dir, 'images')
        self.transforms = transforms
        self.original_shape = original_shape
        self.df = pd.read_csv(csv_path)
        self.image_ids = self.df['ImageId'].unique()

    def rle_decode(self, mask_rle, shape):
        s = str(mask_rle).split()
        starts, lengths = [np.asarray(x, dtype=int) for x in (s[0:][::2], s[1:][::2])]
        starts -= 1
        ends = starts + lengths
        img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
        for lo, hi in zip(starts, ends): img[lo:hi] = 1
        return img.reshape(shape)

    def __len__(self): return len(self.image_ids)

    def __getitem__(self, idx):
        img_name = self.image_ids[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        img = self.apply_preprocessing(img_path)
        img = tv_tensors.Image(img)

        mask_np = np.zeros(self.original_shape, dtype=np.uint8)
        rows = self.df[self.df['ImageId'] == img_name]
        
        for _, row in rows.iterrows():
            rle = row['EncodedPixels']
            if pd.notna(rle) and str(rle).strip().lower() != 'nan':
                class_mask = self.rle_decode(rle, self.original_shape)
                mask_np = np.logical_or(mask_np, class_mask).astype(np.uint8)

        mask = tv_tensors.Mask(mask_np)
        if self.transforms: img, mask = self.transforms(img, mask)
        return {'image': img, 'label': (mask > 0).float()}, img_name

class NormalImageDataset(BaseDefectDataset):
    """Class xử lý dữ liệu Mask dạng ảnh (.jpg, .png, .bmp)"""
    def __init__(self, data_dir, img_size, transforms=None, **kwargs):
        super().__init__(**kwargs)
        self.img_dir = os.path.join(data_dir, 'images')
        self.mask_dir = os.path.join(data_dir, 'masks')
        self.images = sorted(os.listdir(self.img_dir))
        self.transforms = transforms
        self.img_size = img_size
        
    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        img = self.apply_preprocessing(img_path)
        img = tv_tensors.Image(img)

        base_name = os.path.splitext(img_name)[0]
        mask_path = None
        for ext in ['.jpg', '.png', '.bmp', '.tif', '.jpeg']:
            temp_path = os.path.join(self.mask_dir, base_name + ext)
            if os.path.exists(temp_path):
                mask_path = temp_path
                break
        
        if mask_path:
            mask = Image.open(mask_path).convert("L")
        else:
            mask = Image.new('L', (self.img_size, self.img_size), 0)

        mask = tv_tensors.Mask(mask)
        if self.transforms: img, mask = self.transforms(img, mask)
        clean_mask = (mask > 127/255.0).float()
        return {'image': img, 'label': clean_mask}, img_name

def get_dataset(data_dir, format_type, img_size, transforms, csv_path=None, apply_clahe=True, n_channels=1):
    """Trạm định tuyến: Tự động chọn Dataset theo Cấu hình"""
    use_gray = (n_channels == 1)
    if format_type == 'csv':
        return CSVDefectDataset(data_dir, csv_path=csv_path, original_shape=(img_size, img_size), 
                                transforms=transforms, apply_clahe=apply_clahe, use_grayscale=use_gray)
    else:
        return NormalImageDataset(data_dir, img_size=img_size, transforms=transforms, 
                                  apply_clahe=apply_clahe, use_grayscale=use_gray)

In [10]:
# 4. LOSS, METRICS & UTILITIES
# ==========================================
class DiceBCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
    def forward(self, inputs, targets, smooth=1):
        bce_loss = self.bce(inputs, targets)
        inputs = torch.sigmoid(inputs).view(-1)
        targets = targets.view(-1)
        intersection = (inputs * targets).sum()
        dice_loss = 1 - (2. * intersection + smooth) / (inputs.sum() + targets.sum() + smooth)
        return 0.5 * bce_loss + 0.5 * dice_loss
    def _show_dice(self, inputs, targets, smooth=1e-5):
        inputs = (torch.sigmoid(inputs) > 0.5).float().view(-1)
        targets = targets.view(-1)
        intersection = (inputs * targets).sum()
        return ((2. * intersection + smooth) / (inputs.sum() + targets.sum() + smooth)).item()

def iou_on_batch(masks, preds, smooth=1e-5):
    preds = (torch.sigmoid(preds) > 0.5).float().view(-1)
    masks = masks.view(-1)
    intersection = (preds * masks).sum()
    union = preds.sum() + masks.sum() - intersection
    return (intersection + smooth) / (union + smooth).item()

def save_on_batch(images, masks, preds, names, vis_path):
    preds = (torch.sigmoid(preds) > 0.5).float()
    for i in range(images.size(0)):
        save_image(preds[i], os.path.join(vis_path, f"{names[i]}_pred.png"))
        save_image(masks[i], os.path.join(vis_path, f"{names[i]}_gt.png"))

def logger_config(log_path):
    loggerr = logging.getLogger()
    loggerr.setLevel(level=logging.INFO)
    if loggerr.hasHandlers(): loggerr.handlers.clear()
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    handler = logging.FileHandler(log_path, encoding='UTF-8')
    formatter = logging.Formatter('%(asctime)s - %(message)s')
    handler.setFormatter(formatter)
    console = logging.StreamHandler()
    console.setFormatter(formatter)
    loggerr.addHandler(handler)
    loggerr.addHandler(console)
    return loggerr

def save_checkpoint(state, save_path):
    os.makedirs(save_path, exist_ok=True)
    best_model = bool(state.get('best_model', False))
    filename = os.path.join(save_path, f"best_model-{state.get('model', 'model')}.pth.tar") if best_model else os.path.join(save_path, f"model-{state.get('epoch', 0):02d}.pth.tar")
    torch.save(state, filename)

def worker_init_fn(worker_id):
    worker_seed = seed + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)
    torch.manual_seed(worker_seed)

def print_summary(epoch, i, nb_batch, loss, average_loss, batch_time, average_time, iou, average_iou, dice, average_dice, mode, lr, logger):
    summary = f'   [{mode}] Epoch: [{epoch}][{i}/{nb_batch}]  '
    string = f'Loss:{loss:.3f} (Avg {average_loss:.4f}) Dice:{dice:.4f} (Avg {average_dice:.4f}) '
    if mode == 'Train': string += f'LR {lr:.2e}   '
    string += f'(AvgTime {average_time:.1f})'
    logger.info(summary + string)

In [11]:
# 5. VÒNG LẶP HUẤN LUYỆN
# ==========================================
def train_one_epoch(loader, model, criterion, optimizer, writer, epoch, logger):
    logging_mode = 'Train' if model.training else 'Val'
    end = time.time()
    time_sum, loss_sum, dice_sum, iou_sum = 0, 0, 0.0, 0.0

    for i, (sampled_batch, names) in enumerate(loader, 1):
        images, masks = sampled_batch['image'].cuda(), sampled_batch['label'].cuda()
        preds = model(images)
        out_loss = criterion(preds, masks)

        if model.training:
            optimizer.zero_grad()
            out_loss.backward()
            optimizer.step()

        train_iou = iou_on_batch(masks, preds)
        train_dice = criterion._show_dice(preds, masks)
        batch_time = time.time() - end

        if epoch % vis_frequency == 0 and logging_mode == 'Val':
            vis_path = os.path.join(visualize_path, str(epoch))
            os.makedirs(vis_path, exist_ok=True)
            save_on_batch(images, masks, preds, names, vis_path)

        time_sum += len(images) * batch_time
        loss_sum += len(images) * out_loss.item()
        iou_sum += len(images) * train_iou
        dice_sum += len(images) * train_dice

        current_samples = cfg['batch_size'] * (i - 1) + len(images) if i == len(loader) else i * cfg['batch_size']
        avg_loss = loss_sum / current_samples
        avg_time = time_sum / current_samples
        avg_iou = iou_sum / current_samples
        avg_dice = dice_sum / current_samples

        end = time.time()

        if i % print_frequency == 0:
            print_summary(epoch + 1, i, len(loader), out_loss.item(), avg_loss, batch_time, avg_time, 
                          train_iou, avg_iou, train_dice, avg_dice, logging_mode, 
                          min(g["lr"] for g in optimizer.param_groups) if optimizer else 0, logger)

        if writer:
            step = epoch * len(loader) + i
            writer.add_scalar(f'{logging_mode}_Loss', out_loss.item(), step)
            writer.add_scalar(f'{logging_mode}_IoU', train_iou, step)
            writer.add_scalar(f'{logging_mode}_Dice', train_dice, step)

    return avg_loss, avg_dice

def main_loop():
    logger = logger_config(logger_path)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    logger.info(f"[*] Khởi động Training: {CURRENT_TASK} | Batch: {cfg['batch_size']} | Cỡ ảnh: {cfg['img_size']}")
    
    # Chuẩn hóa Normalize động theo V2
    mean = [0.5] if cfg['n_channels'] == 1 else [0.5, 0.5, 0.5]
    std  = [0.5] if cfg['n_channels'] == 1 else [0.5, 0.5, 0.5]

    train_tf = v2.Compose([
        v2.Resize((cfg['img_size'], cfg['img_size']), antialias=True),
        v2.RandomHorizontalFlip(p=0.5),
        v2.RandomVerticalFlip(p=0.5),
        v2.RandomChoice([
            v2.RandomRotation(degrees=[90, 90]),
            v2.RandomRotation(degrees=[180, 180]),
            v2.RandomRotation(degrees=[270, 270]),
        ], p=[0.25, 0.25, 0.25]), 
        v2.RandomRotation(degrees=[-15, 15], interpolation=v2.InterpolationMode.BILINEAR),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=mean, std=std)
    ])

    val_tf = v2.Compose([
        v2.Resize((cfg['img_size'], cfg['img_size']), antialias=True),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=mean, std=std)
    ])

    train_dataset = get_dataset(train_dr, cfg['format'], cfg['img_size'], train_tf, cfg.get('train_csv'), cfg['apply_clahe'], cfg['n_channels'])
    
    val_data_dir = test_dr if cfg['format'] == 'csv' else val_dr
    val_dataset = get_dataset(val_data_dir, cfg['format'], cfg['img_size'], val_tf, cfg.get('val_csv'), cfg['apply_clahe'], cfg['n_channels'])

    train_loader = DataLoader(train_dataset, batch_size=cfg['batch_size'], shuffle=True, worker_init_fn=worker_init_fn, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=cfg['batch_size'], shuffle=False, worker_init_fn=worker_init_fn, num_workers=0)

    model = XULite(img_channels=cfg['n_channels'], num_classes=n_labels, base_c=32).to(device)
    criterion = DiceBCELoss()
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate, weight_decay=weight_decay)
    
    writer = SummaryWriter(tensorboard_folder)
    max_dice = 0.0
    best_epoch = 1
    
    for epoch in range(epochs):
        logger.info(f'\n========= Epoch [{epoch + 1}/{epochs}] =========')
        model.train(True)
        train_one_epoch(train_loader, model, criterion, optimizer, writer, epoch, logger)
        
        logger.info('Validation')
        with torch.no_grad():
            model.eval()
            val_loss, val_dice = train_one_epoch(val_loader, model, criterion, optimizer, writer, epoch, logger)

        if val_dice > max_dice:
            if epoch + 1 > 5:
                logger.info(f'\t Saving best model, mean dice increased from: {max_dice:.4f} to {val_dice:.4f}')
                max_dice = val_dice
                best_epoch = epoch + 1
                save_checkpoint({'epoch': epoch, 'best_model': True, 'model': model_name, 'state_dict': model.state_dict(),
                                 'val_loss': val_loss, 'optimizer': optimizer.state_dict()}, model_path)
        else:
            logger.info(f'\t Mean dice:{val_dice:.4f} does not increase, the best is still: {max_dice:.4f} in epoch {best_epoch}')
            
        early_stopping_count = epoch - best_epoch + 1
        if early_stopping_count > early_stopping_patience:
            logger.info(f'\t early_stopping! Mức chịu đựng: {early_stopping_count}/{early_stopping_patience}')
            break

    writer.close()
    return model

In [20]:
# 1. Chốt Seed hệ thống (Như bạn yêu cầu)
cudnn.benchmark = False
cudnn.deterministic = True
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

os.makedirs(save_path, exist_ok=True)
logger = logger_config(log_path=logger_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"[*] Đang khởi tạo môi trường Test 1 Epoch cho: {task_name}")

# 2. Khởi tạo Pipeline Augmentation
mean = [0.5] if n_channels == 1 else [0.5, 0.5, 0.5]
std  = [0.5] if n_channels == 1 else [0.5, 0.5, 0.5]

train_tf = v2.Compose([
    v2.Resize((img_size, img_size), antialias=True),
    v2.RandomRotation(degrees=[-15, 15], interpolation=v2.InterpolationMode.BILINEAR),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=mean, std=std)
])

# 3. Tự động kiểm tra: Nếu là NEU-seg thì dùng CSV, nếu bộ khác thì dùng Image thông thường
fmt = 'csv' if task_name == 'NEU-seg' else 'image'
logger.info(f"[*] Định dạng dữ liệu nhận diện: {fmt.upper()}")

train_dataset = get_dataset(
    data_dir=train_csv_path, 
    format_type=fmt, 
    img_size=img_size, 
    transforms=train_tf, 
    csv_path=train_csv_path if fmt == 'csv' else None,
    apply_clahe=True, 
    n_channels=n_channels
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, worker_init_fn=worker_init_fn, num_workers=0)

# 4. Khởi tạo Mô hình và Optimizer
model = XULite(img_channels=n_channels, num_classes=n_labels, base_c=32).to(device)
criterion = DiceBCELoss()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate, weight_decay=weight_decay)

# 5. CHẠY HUẤN LUYỆN 1 EPOCH ĐỂ DEBUG
logger.info(f"[*] Bắt đầu Train 1 Epoch...")
model.train(True)

# Lệnh chạy độc lập
avg_loss, avg_dice = train_one_epoch(
    loader=train_loader, 
    model=model, 
    criterion=criterion, 
    optimizer=optimizer, 
    writer=None,  # Tắt tensorboard khi đang test
    epoch=0, 
    logger=logger
)

logger.info(f"[*] TEST HOÀN TẤT! Loss: {avg_loss:.4f} | Dice: {avg_dice:.4f}")

2026-03-20 17:45:06,105 - [*] Đang khởi tạo môi trường Test 1 Epoch cho: NEU-seg
2026-03-20 17:45:06,106 - [*] Định dạng dữ liệu nhận diện: CSV
2026-03-20 17:45:06,106 - [*] CSV path: C:\Users\thuth\Downloads\XUlite\datasets\NEU-seg\TrainingData\NEU_train.csv
2026-03-20 17:45:06,106 - [*] Data directory: C:\Users\thuth\Downloads\XUlite\datasets\NEU-seg\TrainingData
2026-03-20 17:45:06,106 - [*] Image directory: C:\Users\thuth\Downloads\XUlite\datasets\NEU-seg\TrainingData\images


2026-03-20 17:45:06,105 - [*] Đang khởi tạo môi trường Test 1 Epoch cho: NEU-seg
2026-03-20 17:45:06,106 - [*] Định dạng dữ liệu nhận diện: CSV
2026-03-20 17:45:06,106 - [*] CSV path: C:\Users\thuth\Downloads\XUlite\datasets\NEU-seg\TrainingData\NEU_train.csv
2026-03-20 17:45:06,106 - [*] Data directory: C:\Users\thuth\Downloads\XUlite\datasets\NEU-seg\TrainingData
2026-03-20 17:45:06,106 - [*] Image directory: C:\Users\thuth\Downloads\XUlite\datasets\NEU-seg\TrainingData\images


FileNotFoundError: Thiếu thư mục ảnh: C:\Users\thuth\Downloads\XUlite\datasets\NEU-seg\TrainingData\images. Hiện tại chỉ có file CSV nên chưa thể train. Hãy đặt ảnh vào thư mục này hoặc đổi train_image_dir đúng vị trí ảnh.

In [22]:
class SmartDataset(Dataset):
    def __init__(self, data_dir, transforms=None, csv_path=None):
        self.img_dir = os.path.join(data_dir, 'images')
        self.mask_dir = os.path.join(data_dir, 'masks')
        self.images = sorted(os.listdir(self.img_dir))
        self.transforms = transforms
        self.df = pd.read_csv(csv_path) if csv_path and os.path.exists(csv_path) else None

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img = Image.open(os.path.join(self.img_dir, img_name)).convert("L")
        
        # Kiểm tra file ảnh mask
        mask_path = os.path.join(self.mask_dir, img_name)
        if os.path.exists(mask_path):
            mask = Image.open(mask_path).convert("L")
        elif self.df is not None:
            # Logic xử lý CSV (Giả định cột 'id' khớp tên ảnh)
            # Bạn cần viết hàm giải mã CSV phù hợp với định dạng của bạn tại đây
            mask = Image.new('L', (img_size, img_size), 0) 
        else:
            mask = Image.new('L', (img_size, img_size), 0)

        img = tv_tensors.Image(img)
        mask = tv_tensors.Mask(mask)
        if self.transforms: img, mask = self.transforms(img, mask)
        return {'image': img, 'label': (mask > 0).float()}, img_name

class DiceBCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
    def forward(self, inputs, targets, smooth=1):
        bce_loss = self.bce(inputs, targets)
        inputs = torch.sigmoid(inputs).view(-1)
        targets = targets.view(-1)
        intersection = (inputs * targets).sum()
        dice_loss = 1 - (2. * intersection + smooth) / (inputs.sum() + targets.sum() + smooth)
        return 0.5 * bce_loss + 0.5 * dice_loss
    def _show_dice(self, inputs, targets):
        inputs = (torch.sigmoid(inputs) > 0.5).float().view(-1)
        targets = targets.view(-1)
        intersection = (inputs * targets).sum()
        return ((2. * intersection + 1e-5) / (inputs.sum() + targets.sum() + 1e-5)).item()

def iou_on_batch(masks, preds):
    preds = (torch.sigmoid(preds) > 0.5).float().view(-1)
    masks = masks.view(-1)
    intersection = (preds * masks).sum()
    union = preds.sum() + masks.sum() - intersection
    return (intersection + 1e-5) / (union + 1e-5).item()

In [23]:
# Dataset & Transforms (Torchvision v2)
class MoNuSegDataset(Dataset):
    def __init__(self, data_dir, transforms=None):
        self.data_dir = data_dir
        self.transforms = transforms
        
        self.img_dir = os.path.join(data_dir, 'images')
        self.mask_dir = os.path.join(data_dir, 'masks')
        self.images = sorted(os.listdir(self.img_dir))
        
    def __len__(self):
        return len(self.images)
        
    def __getitem__(self, idx):
        img_name = self.images[idx]
        mask_name = img_name 
        
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, mask_name)
        
        img = Image.open(img_path).convert("L") 
        mask = Image.open(mask_path).convert("L")
        
        img = tv_tensors.Image(img)
        mask = tv_tensors.Mask(mask)
        
        if self.transforms is not None:
            img, mask = self.transforms(img, mask)
            
        mask = (mask > 0).float()
        return {'image': img, 'label': mask}, img_name

# Loss & Metrics
class DiceBCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        
    def forward(self, inputs, targets, smooth=1):
        bce_loss = self.bce(inputs, targets)
        inputs = torch.sigmoid(inputs).view(-1)
        targets = targets.view(-1)
        
        intersection = (inputs * targets).sum()
        dice_loss = 1 - (2. * intersection + smooth) / (inputs.sum() + targets.sum() + smooth)
        return 0.5 * bce_loss + 0.5 * dice_loss
        
    def _show_dice(self, inputs, targets, smooth=1e-5):
        inputs = (torch.sigmoid(inputs) > 0.5).float().view(-1)
        targets = targets.view(-1)
        intersection = (inputs * targets).sum()
        dice = (2. * intersection + smooth) / (inputs.sum() + targets.sum() + smooth)
        return dice.item()

def iou_on_batch(masks, preds, smooth=1e-5):
    preds = (torch.sigmoid(preds) > 0.5).float().view(-1)
    masks = masks.view(-1)
    intersection = (preds * masks).sum()
    union = preds.sum() + masks.sum() - intersection
    return (intersection + smooth) / (union + smooth).item()

def save_on_batch(images, masks, preds, names, vis_path):
    preds = (torch.sigmoid(preds) > 0.5).float()
    for i in range(images.size(0)):
        save_image(preds[i], os.path.join(vis_path, f"{names[i]}_pred.png"))
        save_image(masks[i], os.path.join(vis_path, f"{names[i]}_gt.png"))

In [24]:
def logger_config(log_path):
    loggerr = logging.getLogger()
    loggerr.setLevel(level=logging.INFO)
    if getattr(loggerr, "_configured", False):
        return loggerr
    log_dir = os.path.dirname(log_path)
    if log_dir:
        os.makedirs(log_dir, exist_ok=True)
    handler = logging.FileHandler(log_path, encoding='UTF-8')
    formatter = logging.Formatter('%(asctime)s - %(message)s')
    handler.setFormatter(formatter)
    console = logging.StreamHandler()
    console.setFormatter(formatter)
    loggerr.addHandler(handler)
    loggerr.addHandler(console)
    loggerr._configured = True
    return loggerr

def save_checkpoint(state, save_path):
    os.makedirs(save_path, exist_ok=True)
    epoch = int(state.get('epoch', 0))
    best_model = bool(state.get('best_model', False))
    model = str(state.get('model', 'model'))
    
    if best_model:
        filename = os.path.join(save_path, f'best_model-{model}.pth.tar')
    else:
        filename = os.path.join(save_path, f'model-{model}-{epoch:02d}.pth.tar')
    torch.save(state, filename)
    return filename

def worker_init_fn(worker_id):
    worker_seed = seed + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)
    torch.manual_seed(worker_seed)

def print_summary(epoch, i, nb_batch, loss, loss_name, batch_time, average_loss, average_time, iou, average_iou, dice, average_dice, acc, average_acc, mode, lr, logger):
    summary = f'   [{mode}] Epoch: [{epoch}][{i}/{nb_batch}]  '
    string = f'Loss:{loss:.3f} (Avg {average_loss:.4f}) Dice:{dice:.4f} (Avg {average_dice:.4f}) '
    if mode == 'Train':
        string += f'LR {lr:.2e}   '
    string += f'(AvgTime {average_time:.1f})   '
    logger.info(summary + string)

In [25]:
def train_one_epoch(loader, model, criterion, optimizer, writer, epoch, lr_scheduler, model_type, logger):
    logging_mode = 'Train' if model.training else 'Val'
    end = time.time()
    time_sum, loss_sum, dice_sum, iou_sum = 0, 0, 0.0, 0.0

    for i, (sampled_batch, names) in enumerate(loader, 1):
        loss_name = criterion.__class__.__name__
        images, masks = sampled_batch['image'].cuda(), sampled_batch['label'].cuda()

        preds = model(images)
        out_loss = criterion(preds, masks)

        if model.training:
            optimizer.zero_grad()
            out_loss.backward()
            optimizer.step()

        train_iou = iou_on_batch(masks, preds)
        train_dice = criterion._show_dice(preds, masks)
        batch_time = time.time() - end

        if epoch % vis_frequency == 0 and logging_mode == 'Val':
            vis_path = os.path.join(visualize_path, str(epoch))
            os.makedirs(vis_path, exist_ok=True)
            save_on_batch(images, masks, preds, names, vis_path)

        time_sum += len(images) * batch_time
        loss_sum += len(images) * out_loss.item()
        iou_sum += len(images) * train_iou
        dice_sum += len(images) * train_dice

        current_samples = batch_size * (i - 1) + len(images) if i == len(loader) else i * batch_size
        average_loss = loss_sum / current_samples
        average_time = time_sum / current_samples
        train_iou_average = iou_sum / current_samples
        train_dice_avg = dice_sum / current_samples

        end = time.time()

        if i % print_frequency == 0:
            print_summary(epoch + 1, i, len(loader), out_loss.item(), loss_name, batch_time,
                          average_loss, average_time, train_iou, train_iou_average,
                          train_dice, train_dice_avg, 0, 0, logging_mode,
                          lr=min(g["lr"] for g in optimizer.param_groups) if optimizer else 0, logger=logger)

        if writer:
            step = epoch * len(loader) + i
            writer.add_scalar(f'{logging_mode}_{loss_name}', out_loss.item(), step)
            writer.add_scalar(f'{logging_mode}_iou', train_iou, step)
            writer.add_scalar(f'{logging_mode}_dice', train_dice, step)

    if lr_scheduler is not None:
        lr_scheduler.step()
        
    return average_loss, train_dice_avg

def main_loop(batch_size=batch_size, model_type='', tensorboard=True):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    train_tf = v2.Compose([
        v2.Resize((img_size, img_size), antialias=True),
        v2.RandomHorizontalFlip(p=0.5),
        v2.RandomVerticalFlip(p=0.5),
        v2.ToDtype(torch.float32, scale=True),
    ])

    val_tf = v2.Compose([
        v2.Resize((img_size, img_size), antialias=True),
        v2.ToDtype(torch.float32, scale=True),
    ])

    train_dataset = MoNuSegDataset(train_dr, transforms=train_tf)
    val_dataset = MoNuSegDataset(val_dr, transforms=val_tf)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              worker_init_fn=worker_init_fn, num_workers=0, pin_memory=False)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                            worker_init_fn=worker_init_fn, num_workers=0, pin_memory=False)

    logger.info(f"Selected Model: {model_type}")

    if model_type == 'XULite':
        model = XULite(img_channels=n_channels, num_classes=n_labels, base_c=32)
    else:
        raise TypeError('Please enter a valid model type')

    model = model.to(device)
    criterion = DiceBCELoss()
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate)
    lr_scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=1, eta_min=1e-4) if cosineLR else None
        
    writer = SummaryWriter(tensorboard_folder) if tensorboard else None

    max_dice = 0.0
    best_epoch = 1
    
    for epoch in range(epochs):
        logger.info(f'\n========= Epoch [{epoch + 1}/{epochs}] =========')
        model.train(True)
        train_one_epoch(train_loader, model, criterion, optimizer, writer, epoch, None, model_type, logger)
        
        logger.info('Validation')
        with torch.no_grad():
            model.eval()
            val_loss, val_dice = train_one_epoch(val_loader, model, criterion, optimizer, writer, epoch, lr_scheduler, model_type, logger)

        if val_dice > max_dice:
            if epoch + 1 > 5:
                logger.info(f'\t Saving best model, mean dice increased from: {max_dice:.4f} to {val_dice:.4f}')
                max_dice = val_dice
                best_epoch = epoch + 1
                save_checkpoint({'epoch': epoch, 'best_model': True, 'model': model_type, 'state_dict': model.state_dict(),
                                 'val_loss': val_loss, 'optimizer': optimizer.state_dict()}, model_path)
        else:
            logger.info(f'\t Mean dice:{val_dice:.4f} does not increase, the best is still: {max_dice:.4f} in epoch {best_epoch}')
            
        early_stopping_count = epoch - best_epoch + 1
        logger.info(f'\t early_stopping_count: {early_stopping_count}/{early_stopping_patience}')

        if early_stopping_count > early_stopping_patience:
            logger.info('\t early_stopping!')
            break

    if writer:
        writer.close()
    return model

In [ ]:
# 1. Lấy đường dẫn file weights tốt nhất vừa lưu
best_weight_path = os.path.join(model_path, 'best_model-XULite.pth.tar')

# Thay 'image_01.png' bằng tên một file thực tế trong TestData của bạn
test_img_name = 'image_01.png'
test_img_path = os.path.join(test_dr, 'images', test_img_name) 
test_mask_path = os.path.join(test_dr, 'masks', test_img_name) 

# 2. Khởi tạo mô hình và nạp trọng số (weights)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_pred = XULite(img_channels=1, num_classes=1, base_c=32).to(device)

if os.path.exists(best_weight_path):
    checkpoint = torch.load(best_weight_path, map_location=device)
    model_pred.load_state_dict(checkpoint['state_dict'])
    print("Đã load weights thành công!")
else:
    print(f"Lỗi: Không tìm thấy file {best_weight_path}")

# 3. Tiền xử lý (Giống hệt lúc Val)
predict_tf = v2.Compose([
    v2.Resize((img_size, img_size), antialias=True),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
])

img_tensor = predict_tf(Image.open(test_img_path).convert("L"))
input_batch = img_tensor.unsqueeze(0).to(device) 

# 4. Suy luận (Predict)
model_pred.eval()
with torch.no_grad():
    output = model_pred(input_batch)
    pred_mask = (torch.sigmoid(output) > 0.5).float()

pred_mask_numpy = pred_mask.squeeze().cpu().numpy()
img_numpy = img_tensor.squeeze().cpu().numpy()

# 5. Vẽ hình lên Jupyter
fig, axes = plt.subplots(1, 3 if os.path.exists(test_mask_path) else 2, figsize=(15, 5))

axes[0].imshow(img_numpy, cmap='gray')
axes[0].set_title('Ảnh đầu vào (Input)')
axes[0].axis('off')

axes[1].imshow(pred_mask_numpy, cmap='gray')
axes[1].set_title('Mô hình Dự đoán (Pred)')
axes[1].axis('off')

if os.path.exists(test_mask_path):
    gt_mask = predict_tf(Image.open(test_mask_path).convert("L")).squeeze().numpy()
    axes[2].imshow((gt_mask > 0).astype(float), cmap='gray')
    axes[2].set_title('Đáp án chuẩn (Ground Truth)')
    axes[2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
class AxialDW(nn.Module):
    def __init__(self, dim, mixer_kernel, dilation = 1):
        super().__init__()
        h, w = mixer_kernel
        self.dw_h = nn.Conv2d(dim, dim, kernel_size=(h, 1), padding='same', groups = dim, dilation = dilation)
        self.dw_w = nn.Conv2d(dim, dim, kernel_size=(1, w), padding='same', groups = dim, dilation = dilation)

    def forward(self, x):
        x = x + self.dw_h(x) + self.dw_w(x)
        return x


In [ ]:
class EncoderBlock(nn.Module):
    """Encoding then downsampling"""
    def __init__(self, in_c, out_c, mixer_kernel = (7, 7)):
        super().__init__()
        self.dw = AxialDW(in_c, mixer_kernel = (7, 7))
        self.bn = nn.BatchNorm2d(in_c)
        self.pw = nn.Conv2d(in_c, out_c, kernel_size=1)
        self.down = nn.MaxPool2d((2,2))
        self.act = nn.GELU()

    def forward(self, x):
        skip = self.bn(self.dw(x))
        x = self.act(self.down(self.pw(skip)))
        return x, skip
    

class XConv(nn.Module):
    """Khối 7x7 XConv (Sử dụng Depthwise Convolution để tối ưu tham số)"""
    def __init__(self, in_c, kernel_size=7):
        super().__init__()
        # padding = 3 để giữ nguyên kích thước không gian (H, W), bắt buộc để có thể cộng Residual (+ x)
        self.conv = nn.Conv2d(
            in_c, in_c, 
            kernel_size=kernel_size, 
            padding=kernel_size // 2, 
            groups=in_c,  # <-- Depthwise
            bias=False
        )

    def forward(self, x):
        return self.conv(x)

class EncoderBlock(nn.Module):
    """Encoding then downsampling"""
    def __init__(self, in_c, out_c):
        super().__init__()
        self.xconv = XConv(in_c, kernel_size=7)
        self.bn = nn.BatchNorm2d(in_c)
        self.pw = nn.Conv2d(in_c, out_c, kernel_size=1, bias=False)
        self.down = nn.MaxPool2d(kernel_size=2, stride=2)
        self.act = nn.GELU()

    def forward(self, x):
        # xconv(x) là nhánh XConv 7x7
        # + x là nhánh Residual
        # Tổng đi qua Batch Norm và được giữ lại làm skip connection
        skip = self.bn(self.xconv(x) + x)
        
        # Dữ liệu tiếp tục qua PW conv -> Max Pooling -> GELU
        out = self.act(self.down(self.pw(skip)))
        
        return out, skip

In [ ]:
# Giả lập một batch gồm 2 ảnh xám kích thước 256x256
dummy_input = torch.randn(2, 1, 256, 256)
model = XULite(img_channels=1, num_classes=1, base_c=32)

output = model(dummy_input)
print(f"Kích thước ảnh đầu vào: {dummy_input.shape}")
print(f"Kích thước Mask dự đoán:{output.shape}")

In [ ]:
class DecoderBlock(nn.Module):
    """Decoding, Upsampling and Feature Fusion"""
    def __init__(self, in_c, skip_c, out_c):
        """
        in_c: Số kênh đầu vào từ lớp dưới đi lên
        skip_c: Số kênh của skip connection đi ngang sang từ Encoder
        out_c: Số kênh đầu ra mong muốn
        """
        super().__init__()
        
        # 1. UpSampling (Phóng to kích thước H, W lên gấp đôi)
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        
        # 2. Batch Normalization (Áp dụng ngay sau UpSampling theo sơ đồ)
        self.bn = nn.BatchNorm2d(in_c)
        
        # 3. PW conv 1 (Pointwise Convolution đầu tiên)
        # Đầu vào là tổng số kênh sau khi Concat (in_c + skip_c)
        self.pw1 = nn.Conv2d(in_c + skip_c, out_c, kernel_size=1, bias=False)
        
        # 4. 7x7 XConv
        self.xconv = XConv(out_c, kernel_size=7)
        
        # 5. PW conv 2 (Pointwise Convolution thứ hai)
        self.pw2 = nn.Conv2d(out_c, out_c, kernel_size=1, bias=False)
        
        # 6. GELU Activation
        self.act = nn.GELU()

    def forward(self, x, skip):
        # Bước 1 & 2: UpSampling -> BN
        x = self.bn(self.up(x))
        
        # Bước 3: Concat với skip connection
        # dim=1 là nối theo chiều channel (B, C, H, W)
        x = torch.cat([x, skip], dim=1)
        
        # Bước 4: Qua lớp PW conv đầu tiên
        x = self.pw1(x)
        
        # Bước 5: 7x7 XConv + Residual (Cộng x ngay sau xconv)
        x = self.xconv(x) + x
        
        # Bước 6: Qua lớp PW conv thứ hai -> GELU
        out = self.act(self.pw2(x))
        
        return out

In [ ]:
class XConv(nn.Module):
    """Khối 7x7 XConv (Depthwise)"""
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=7, padding=3, groups=channels, bias=False)

    def forward(self, x):
        return self.conv(x)

class AxialDW(nn.Module):
    """Khối AxialDW 1x7 + 7x1 (Cộng song song 2 nhánh)"""
    def __init__(self, channels):
        super().__init__()
        self.dw_1x7 = nn.Conv2d(channels, channels, kernel_size=(1, 7), padding=(0, 3), groups=channels, bias=False)
        self.dw_7x1 = nn.Conv2d(channels, channels, kernel_size=(7, 1), padding=(3, 0), groups=channels, bias=False)

    def forward(self, x):
        # Cộng song song đặc trưng theo chiều dọc và ngang
        return self.dw_1x7(x) + self.dw_7x1(x)

class MultiScaleFusionBlock(nn.Module):
    """Khối trích xuất đặc trưng đa tỷ lệ (Sơ đồ d)"""
    def __init__(self, in_c, out_c):
        super().__init__()
        
        # 1. Lớp PW conv đầu tiên
        # Giả sử ta giữ nguyên số kênh trong các nhánh để tiết kiệm tham số (mid_c = out_c)
        mid_c = out_c 
        self.pw1 = nn.Conv2d(in_c, mid_c, kernel_size=1, bias=False)
        
        # 2. Bốn nhánh song song (4 parallel branches)
        # Nhánh 1: 7x7 XConv
        self.branch1 = XConv(mid_c)
        
        # Nhánh 2: AxialDW 1x7 + 7x1
        self.branch2 = AxialDW(mid_c)
        
        # Nhánh 3: 3x3 DWconv
        self.branch3 = nn.Conv2d(mid_c, mid_c, kernel_size=3, padding=1, groups=mid_c, bias=False)
        
        # Nhánh 4: Identity (Đường đi thẳng, không cần định nghĩa layer)

        # 3. Lớp BN áp dụng sau khi Concat
        # Có 4 nhánh, mỗi nhánh có số kênh là mid_c -> Tổng số kênh sau concat là mid_c * 4
        self.bn = nn.BatchNorm2d(mid_c * 4)
        
        # 4. Lớp PW conv cuối cùng để nén số kênh lại
        self.pw2 = nn.Conv2d(mid_c * 4, out_c, kernel_size=1, bias=False)
        
        # 5. Hàm kích hoạt GELU
        self.act = nn.GELU()

    def forward(self, x):
        # Đi qua PW conv đầu tiên
        x = self.pw1(x)
        
        # Chia làm 4 nhánh
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = x  # Đường thẳng (Identity)
        
        # Concat 4 nhánh lại với nhau (dim=1 là chiều channel)
        out = torch.cat([b1, b2, b3, b4], dim=1)
        
        # Qua BN -> PW conv -> GELU
        out = self.bn(out)
        out = self.pw2(out)
        out = self.act(out)
        
        return out

In [ ]:
class XConv7x7(nn.Module):
    """Khối tích chập hình chữ X kích thước 7x7"""
    def __init__(self, channels):
        super().__init__()
        
        # 1. Khởi tạo một lớp Depthwise Convolution 7x7 bình thường
        self.conv = nn.Conv2d(
            channels, channels, 
            kernel_size=7, 
            padding=3, 
            groups=channels, 
            bias=False
        )
        
        # 2. Tạo mặt nạ (Mask) hình chữ X
        # Khởi tạo ma trận toàn số 0 kích thước 7x7
        mask = torch.zeros(7, 7)
        for i in range(7):
            mask[i, i] = 1         # Đường chéo chính (Màu xanh lam trong hình)
            mask[i, 6 - i] = 1     # Đường chéo phụ (Màu đỏ trong hình)
            # Pixel ở giữa (i=3) sẽ tự động chia sẻ do bị ghi đè thành 1 (Màu tím)
            
        # 3. Định hình lại Mask để khớp với kích thước của trọng số (Weight)
        # Weight của Depthwise Conv có dạng: (channels, 1, kernel_H, kernel_W)
        mask = mask.view(1, 1, 7, 7).expand(channels, 1, 7, 7)
        
        # 4. Đăng ký Mask vào buffer
        # Lệnh này giúp PyTorch hiểu mask là một tensor gắn liền với mô hình 
        # (sẽ tự động di chuyển lên GPU cùng model) nhưng KHÔNG bị cập nhật bởi thuật toán tối ưu (Optimizer)
        self.register_buffer('mask', mask)

    def forward(self, x):
        # Nhân trọng số của mạng với mặt nạ hình chữ X
        # Các pixel nằm ngoài chữ X sẽ bị nhân với 0 (loại bỏ)
        masked_weight = self.conv.weight * self.mask
        
        # Sử dụng F.conv2d để thực hiện phép chập với trọng số mới đã bị che mask
        out = F.conv2d(
            x, 
            weight=masked_weight, 
            bias=self.conv.bias, 
            stride=self.conv.stride, 
            padding=self.conv.padding, 
            groups=self.conv.groups
        )
        
        return out